# Prototype

In [1]:
import time
from collections.abc import Iterable
from copy import copy, deepcopy
from typing import Any, Protocol, Self, runtime_checkable

## Intro

Prototype is a creational design pattern. It allows to **create** the copies of some class instance.

A class can be made copyable or deep-copyable via the corresponding dunder (double underscore) methods [\_\_copy\_\_](https://docs.python.org/3/library/copy.html#object.__copy__) and [\_\_deepcopy\_\_](https://docs.python.org/3/library/copy.html#object.__deepcopy__).

In [2]:
class MyCollection:
    def __init__(self, it: Iterable) -> None:
        self._it = it

    def __repr__(self) -> str:
        cls_name = type(self).__name__
        return f"{cls_name}(it={self._it})"

    def __copy__(self) -> Self:
        cls = type(self)
        # shallow copy
        return cls(copy(self._it))  # or self._it.__copy__()

    def __deepcopy__(self, memo) -> Self:
        cls = type(self)
        # deep (true) copy
        return cls(deepcopy(self._it))  # or self._it.__deepcopy__()


lst = [1, [3, 5], [8, [9, [10]]]]


mc = MyCollection(lst)

mc_copy = copy(mc)
assert isinstance(mc_copy, MyCollection)

mc_deepcopy = deepcopy(mc)
assert isinstance(mc_deepcopy, MyCollection)

# Oops
lst[-1][-1] = [21, [42]]

print(f"List = {lst}")
print(f"MyCollection = {mc}")
print(f"Shallow MyCollection Copy: {mc_copy}")
print(f"Deep MyCollection Copy: {mc_deepcopy}")

List = [1, [3, 5], [8, [21, [42]]]]
MyCollection = MyCollection(it=[1, [3, 5], [8, [21, [42]]]])
Shallow MyCollection Copy: MyCollection(it=[1, [3, 5], [8, [21, [42]]]])
Deep MyCollection Copy: MyCollection(it=[1, [3, 5], [8, [9, [10]]]])


We can also define so-called "Cloneable" protocols. They will be defined via the "structural subtyping" technique.

From [docs](https://typing.python.org/en/latest/reference/protocols.html):

> The Python type system supports two ways of deciding whether two objects are compatible as types: nominal subtyping and structural subtyping.
>
> Nominal subtyping is strictly based on the class hierarchy. If class Dog inherits class Animal, it’s a subtype of Animal. Instances of Dog can be used when Animal instances are expected. This form of subtyping is what Python’s type system predominantly uses: it’s easy to understand and produces clear and concise error messages, and matches how the native isinstance check works – based on class hierarchy.
>
> Structural subtyping is based on the operations that can be performed with an object. Class Dog is a structural subtype of class Animal if the former has all attributes and methods of the latter, and with compatible types.
>
> Structural subtyping can be seen as a static equivalent of duck typing, which is well known to Python programmers. See PEP 544 for the detailed specification of protocols and structural subtyping in Python.

In [3]:
# for `isinstance` checks
@runtime_checkable
class SupportsClone(Protocol):  # from typing
    def clone(self) -> Self: ...  # the body does not matter for protocols


@runtime_checkable
class SupportsDeepClone(Protocol):
    def deepclone(self) -> Self: ...


# A class to be a Protocol must explicitly inherit the typing.Protocol class
# Just inheriting protocol classes does not make this class a Protocol. 
@runtime_checkable
class Cloneable(SupportsClone, SupportsDeepClone, Protocol):
    ...

Time to play a bit with `Protocol`s and structural subtyping.

In [4]:
class BaseVirus:  # no inheritance here!
    def __init__(self, *args, **kwargs) -> None:
        self._args = args
        self._kwargs = kwargs


class CloneableVirus(BaseVirus):
    def clone(self):
        cls = type(self)
        return cls(*copy(self._args), **copy(self._kwargs))


class DeeplyCloneableVirus(BaseVirus):
    def deepclone(self):
        cls = type(self)
        return cls(*deepcopy(self._args), **deepcopy(self._kwargs))


cp_virus = CloneableVirus()  # no care about incoming (kw)args
deepcp_virus = DeeplyCloneableVirus()

# no inheritance, but recognisable -> thanks to substructural subTYPing

cp_virus_copy = cp_virus.clone()
assert isinstance(cp_virus_copy, SupportsClone)  # there is the clone() method
assert not isinstance(cp_virus_copy, SupportsDeepClone)  # there is no deepclone() method

deepcp_virus_copy = deepcp_virus.deepclone()
assert isinstance(deepcp_virus_copy, SupportsDeepClone)  # there is the deepclone() method
assert not isinstance(deepcp_virus_copy, SupportsClone)  # there is no clone() method

## Why care?

The so-called examples above tell nothing about benefits of this pattern and use cases. One reason, and perhaps the main one, is to make the objects of a certain class when:

- creating a new object (via the "\_\_init\_\_" or defined [@classmethod](https://docs.python.org/3/library/functions.html#classmethod)s) is expensive AND
- copying is cheap

In [5]:
class ComplexResource:
    def __init__(self, res1: Any = None, res2: Any = None) -> None:
        self._sub1 = res1
        if not res1:
            # costs
            time.sleep(1)
            self._sub1 = 21
            print(f"subresource_1 is created: {self._sub1}")
        self._sub2 = res2
        if not res2:
            # costs
            time.sleep(1)
            self._sub2 = 42
            print(f"subresource_2 is created: {self._sub2}")

    def __repr__(self) -> str:
        cls_name = type(self).__name__
        res1 = self._sub1
        res2 = self._sub2
        return f"{cls_name}(res1={res1}, res2={res2})"

    def clone(self) -> Self:
        cls = type(self)
        return cls(res1=self._sub1, res2=self._sub2)

# time-consuming initialisation
resource = ComplexResource()

# almost zero-cost instantiation
resources = [resource] + [resource.clone() for _ in range(10)]
print(f"Resources = {resources}")

subresource_1 is created: 21
subresource_2 is created: 42
Resources = [ComplexResource(res1=21, res2=42), ComplexResource(res1=21, res2=42), ComplexResource(res1=21, res2=42), ComplexResource(res1=21, res2=42), ComplexResource(res1=21, res2=42), ComplexResource(res1=21, res2=42), ComplexResource(res1=21, res2=42), ComplexResource(res1=21, res2=42), ComplexResource(res1=21, res2=42), ComplexResource(res1=21, res2=42), ComplexResource(res1=21, res2=42)]


## References

- https://typing.python.org/en/latest/spec/protocol.html
- https://typing.python.org/en/latest/reference/protocols.html
- https://mypy.readthedocs.io/en/stable/protocols.html